# Multi-Provider Agentic Pipeline: Coastal City Resilience Plan

[Workflow design Patterns](https://claude.ai/share/de732577-430c-467d-a87e-c8f8e2e999e1)

!["Coastal City Resilience Plan"](./combined_pipeline_resilience.svg)

This notebook demonstrates **four agentic design patterns** composed into a single pipeline, dispatching work across **three LLM providers** (Anthropic, OpenAI, Google).

### Pipeline architecture

```
Phase 1 - Prompt Chaining      | Parse scenario -> structured context        | Claude Sonnet 4
Phase 2 - Parallelization      | 4 priority areas run concurrently           | Gemini 2.5 Pro, Opus 4, o3, GPT-4o
Phase 3 - Prompt Chaining      | Synthesize + alternatives & triggers        | Claude Opus 4
Phase 4 - Evaluator-Optimizer  | Rubric-based quality loop (max 3 iters)     | GPT-4o
```

### Model assignments

| Phase | Role | Model | Provider | Rationale |
|-------|------|-------|----------|-----------|
| 1 | Parse scenario | Claude Sonnet 4 | Anthropic | Structured extraction |
| 2a | Emergency prep | Gemini 2.5 Pro | Google | Strong structured planning |
| 2b | Infrastructure adaptation | Claude Opus 4 | Anthropic | Nuanced tradeoff reasoning |
| 2c | Managed retreat | OpenAI o3 | OpenAI | Deep ethical reasoning |
| 2d | Fiscal sustainability | GPT-4o | OpenAI | Cost modeling |
| 3 | Synthesize & alternatives | Claude Opus 4 | Anthropic | Reconciling conflicting plans |
| 4 | Evaluator | GPT-4o | OpenAI | Independent provider from synthesizer |

## 1. Setup

In [ ]:
# %pip install httpx -q

In [1]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

# Uncomment and fill in if not already set in your environment:
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."
# os.environ["OPENAI_API_KEY"]    = "sk-..."
# os.environ["GOOGLE_API_KEY"]    = "..."

required = {
    "ANTHROPIC_API_KEY": "Anthropic (Claude Sonnet 4, Opus 4)",
    "OPENAI_API_KEY":    "OpenAI (GPT-4o, o3)",
    "GOOGLE_API_KEY":    "Google (Gemini 2.5 Pro)",
}
for key, desc in required.items():
    status = "\u2705" if os.environ.get(key) else "\u274c MISSING"
    print(f"  {status}  {key} \u2192 {desc}")

  ✅  ANTHROPIC_API_KEY → Anthropic (Claude Sonnet 4, Opus 4)
  ✅  OPENAI_API_KEY → OpenAI (GPT-4o, o3)
  ✅  GOOGLE_API_KEY → Google (Gemini 2.5 Pro)


## 2. Scenario prompt

This is the complex, multi-dimensional question we send through the pipeline.

In [2]:
SCENARIO = (
    "You are advising the mayor of a mid-sized coastal city (population ~500,000) where "
    "30% of developed land lies in the floodplain; median sea-level rise projections are "
    "0.6 m by 2050 with a 1.2 m worst-case; the city faces a housing affordability crisis, "
    "aging infrastructure, and limited fiscal capacity (annual operating budget ~$1.5B, "
    "capital budget ~$200M/yr), and a recent storm caused $1.8B in damage that "
    "disproportionately harmed low-income neighborhoods\u2014devise a prioritized, evidence-based "
    "30-year resilience-and-equity plan that balances (1) immediate emergency preparedness, "
    "(2) medium-term infrastructure adaptation (including tradeoffs between nature-based and "
    "grey solutions), (3) long-term managed retreat and land-use transformation, and "
    "(4) fiscal sustainability; for each priority specify concrete interventions, "
    "order-of-magnitude cost estimates, realistic timelines, funding and financing mechanisms, "
    "governance and community-engagement structures, measurable success metrics, and the main "
    "social/ethical trade-offs; explicitly list your critical assumptions and quantify "
    "uncertainties where possible, describe at least two credible alternative strategies you "
    "considered and why you rejected them, provide five leading indicators that would trigger "
    "switching to an alternative strategy, and state which local datasets you would request "
    "to reduce key uncertainties and how each dataset would likely change your recommendations."
)

print(f"Scenario length: {len(SCENARIO):,} characters")
print(f"Approximate tokens: ~{len(SCENARIO) // 4:,}")

Scenario length: 1,445 characters
Approximate tokens: ~361


## 3. Configuration & model registry

All model assignments live in one registry. Swap any model by changing a single line.

In [3]:
from __future__ import annotations

import asyncio
import json
import time
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from enum import Enum
from typing import Any

import httpx

MAX_EVAL_ITERATIONS = 3
EVAL_PASS_THRESHOLD = 7
HTTP_TIMEOUT = 180.0


class Provider(Enum):
    ANTHROPIC = "anthropic"
    OPENAI = "openai"
    GOOGLE = "google"


@dataclass
class ModelConfig:
    provider: Provider
    model_id: str
    display_name: str
    max_tokens: int = 4096
    temperature: float = 0.4


MODELS = {
    "parse":      ModelConfig(Provider.ANTHROPIC, "claude-sonnet-4-20250514", "Claude Sonnet 4"),
    "emergency":  ModelConfig(Provider.GOOGLE,    "gemini-2.5-pro",           "Gemini 2.5 Pro"),
    "adaptation": ModelConfig(Provider.ANTHROPIC, "claude-opus-4-20250514",   "Claude Opus 4"),
    "retreat":    ModelConfig(Provider.OPENAI,    "o3",                       "OpenAI o3", temperature=1.0),
    "fiscal":     ModelConfig(Provider.OPENAI,    "gpt-4o",                   "GPT-4o"),
    "synthesize": ModelConfig(Provider.ANTHROPIC, "claude-opus-4-20250514",   "Claude Opus 4"),
    "evaluator":  ModelConfig(Provider.OPENAI,    "gpt-4o",                   "GPT-4o"),
}

print("Model Registry")
print("\u2500" * 72)
for role, cfg in MODELS.items():
    print(f"  {role:14s} \u2502 {cfg.display_name:20s} \u2502 {cfg.provider.value}")

Model Registry
────────────────────────────────────────────────────────────────────────
  parse          │ Claude Sonnet 4      │ anthropic
  emergency      │ Gemini 2.5 Pro       │ google
  adaptation     │ Claude Opus 4        │ anthropic
  retreat        │ OpenAI o3            │ openai
  fiscal         │ GPT-4o               │ openai
  synthesize     │ Claude Opus 4        │ anthropic
  evaluator      │ GPT-4o               │ openai


## 4. LLM provider clients

Each provider has its own API shape. The `call_llm()` dispatcher routes by role automatically.

In [4]:
class LLMClient(ABC):
    @abstractmethod
    async def complete(self, client: httpx.AsyncClient, system: str, user: str, config: ModelConfig) -> str: ...


class AnthropicClient(LLMClient):
    API_URL = "https://api.anthropic.com/v1/messages"

    async def complete(self, client, system, user, config):
        resp = await client.post(
            self.API_URL,
            headers={
                "x-api-key": os.environ["ANTHROPIC_API_KEY"],
                "anthropic-version": "2023-06-01",
                "content-type": "application/json",
            },
            json={
                "model": config.model_id,
                "max_tokens": config.max_tokens,
                "temperature": config.temperature,
                "system": system,
                "messages": [{"role": "user", "content": user}],
            },
        )
        resp.raise_for_status()
        data = resp.json()
        return "".join(b["text"] for b in data["content"] if b["type"] == "text")


class OpenAIClient(LLMClient):
    API_URL = "https://api.openai.com/v1/chat/completions"

    async def complete(self, client, system, user, config):
        payload: dict[str, Any] = {
            "model": config.model_id,
            "messages": [
                {"role": "system", "content": system},
                {"role": "user", "content": user},
            ],
        }
        if config.model_id.startswith("o"):
            payload["max_completion_tokens"] = config.max_tokens
        else:
            payload["max_tokens"] = config.max_tokens
            payload["temperature"] = config.temperature

        resp = await client.post(
            self.API_URL,
            headers={
                "Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}",
                "Content-Type": "application/json",
            },
            json=payload,
        )
        resp.raise_for_status()
        return resp.json()["choices"][0]["message"]["content"]


class GoogleClient(LLMClient):
    API_BASE = "https://generativelanguage.googleapis.com/v1beta/models"

    async def complete(self, client, system, user, config):
        url = f"{self.API_BASE}/{config.model_id}:generateContent"
        resp = await client.post(
            url,
            params={"key": os.environ["GOOGLE_API_KEY"]},
            json={
                "system_instruction": {"parts": [{"text": system}]},
                "contents": [{"role": "user", "parts": [{"text": user}]}],
                "generationConfig": {
                    "temperature": config.temperature,
                    "maxOutputTokens": config.max_tokens,
                },
            },
        )
        resp.raise_for_status()
        return resp.json()["candidates"][0]["content"]["parts"][0]["text"]


_CLIENTS: dict[Provider, LLMClient] = {
    Provider.ANTHROPIC: AnthropicClient(),
    Provider.OPENAI:    OpenAIClient(),
    Provider.GOOGLE:    GoogleClient(),
}


async def call_llm(client: httpx.AsyncClient, role: str, system: str, user: str) -> str:
    config = MODELS[role]
    llm_client = _CLIENTS[config.provider]
    print(f"  \u23f3 [{role}] Calling {config.display_name}...")
    start = time.perf_counter()
    result = await llm_client.complete(client, system, user, config)
    elapsed = time.perf_counter() - start
    print(f"  \u2705 [{role}] {config.display_name} responded ({elapsed:.1f}s, {len(result):,} chars)")
    return result

print("\u2705 Provider clients ready")

✅ Provider clients ready


## 5. Prompt templates

Each phase has carefully scoped prompts. Key principle: **each LLM call gets only what it needs.**

In [6]:
PHASE1_SYSTEM = (
    "You are a senior urban resilience analyst. Parse the provided "
    "city scenario into a structured JSON context object. Extract:\n"
    "- city_profile: population, floodplain_pct, budget_operating, budget_capital\n"
    "- climate_risk: sea_level_median_2050, sea_level_worst_case, recent_storm_damage\n"
    "- social_context: housing_crisis (bool), aging_infrastructure (bool), "
    "equity_concerns (description)\n"
    "- critical_assumptions: list of 8-12 assumptions not stated in the scenario "
    "(e.g., discount rate, population growth, insurance penetration)\n"
    "- key_uncertainties: list of 5-8 quantified uncertainties with ranges\n\n"
    "Return ONLY valid JSON, no markdown fences, no commentary."
)


def make_priority_prompt(priority: str, description: str, context_json: str) -> tuple[str, str]:
    system = (
        f"You are a specialist in {description}. You are contributing one "
        f'section of a 30-year coastal city resilience plan.\n\n'
        f'Given the parsed city context below, produce a detailed plan for the '
        f'"{priority}" priority area. Include:\n'
        "1. Concrete interventions (3-5) with order-of-magnitude cost estimates\n"
        "2. Realistic timelines (immediate / 5yr / 10yr / 30yr)\n"
        "3. Funding and financing mechanisms specific to this area\n"
        "4. Governance and community engagement structures\n"
        "5. Measurable success metrics (quantitative where possible)\n"
        "6. Main social/ethical tradeoffs for this priority area\n\n"
        "IMPORTANT CONSTRAINTS:\n"
        "- Total capital costs across ALL four priorities must stay within ~$200M/yr. "
        "Assume your area gets roughly 25-35% of that unless you argue otherwise.\n"
        "- All interventions must address equity \u2014 explain how low-income neighborhoods "
        "benefit specifically.\n"
        '- Be concrete: "$50M" not "significant investment"; "2027" not "near-term".\n\n'
        "Respond in well-structured markdown."
    )
    user = f"City context:\n{context_json}"
    return system, user


PRIORITY_CONFIGS = {
    "emergency":  ("Immediate emergency preparedness",
                   "disaster response, early warning systems, and emergency management"),
    "adaptation": ("Medium-term infrastructure adaptation",
                   "infrastructure engineering, nature-based solutions, and grey infrastructure"),
    "retreat":    ("Long-term managed retreat and land-use transformation",
                   "urban planning, managed retreat policy, and environmental justice"),
    "fiscal":     ("Fiscal sustainability",
                   "municipal finance, public budgeting, and infrastructure funding mechanisms"),
}


SYNTHESIS_SYSTEM = (
    "You are the lead advisor synthesizing four priority-area plans "
    "into a single coherent 30-year resilience strategy. You will receive the parsed city "
    "context and four specialist plans.\n\n"
    "Your tasks:\n"
    "1. RESOLVE BUDGET CONFLICTS: The four plans likely exceed the $200M/yr cap. "
    "Prioritize, phase, and cut until totals are feasible. Show your math.\n"
    "2. IDENTIFY CROSS-CUTTING SYNERGIES: Where do investments serve multiple priorities?\n"
    "3. PRODUCE A UNIFIED TIMELINE: Merge into a single phased roadmap.\n"
    "4. GENERATE ALTERNATIVES: Describe at least 2 credible alternative strategies "
    "you considered and why you rejected them.\n"
    "5. DEFINE 5 LEADING INDICATORS that would trigger switching to an alternative.\n"
    "6. LIST LOCAL DATASETS to reduce key uncertainties, and how each "
    "would likely change your recommendations.\n\n"
    "Format as a comprehensive markdown report with clear section headers."
)


EVALUATOR_SYSTEM = (
    "You are a rigorous quality evaluator for urban resilience plans. "
    "Score the plan against this rubric. For each check, respond PASS or FAIL.\n\n"
    "RUBRIC:\n"
    "1. BUDGET_FEASIBILITY: Do total capital costs stay within ~$200M/yr?\n"
    "2. EQUITY_SPECIFICITY: Are equity impacts specific (named neighborhoods, "
    "income thresholds, %) rather than vague?\n"
    "3. COST_CONCRETENESS: Are cost estimates order-of-magnitude numbers ($XM)?\n"
    "4. TIMELINE_CONCRETENESS: Are timelines specific years (2027, 2035)?\n"
    "5. ALTERNATIVES_PRESENT: Are 2+ genuinely different alternative strategies described?\n"
    "6. LEADING_INDICATORS: Are 5 leading indicators defined with quantitative thresholds?\n"
    "7. DATASETS_LISTED: Are local datasets specified with impact on recommendations?\n"
    "8. ASSUMPTIONS_EXPLICIT: Are assumptions listed and uncertainties quantified?\n\n"
    "After scoring, provide SPECIFIC FEEDBACK for each FAIL item.\n\n"
    "Respond as JSON:\n"
    '{"checks": {"BUDGET_FEASIBILITY": {"pass": true, "reason": "..."}, ...}, '
    '"total_passed": N, "feedback": "Specific actionable feedback..."}\n'
    "Return ONLY valid JSON."
)

print("\u2705 Prompt templates defined")
print(f"   Phase 1 system: {len(PHASE1_SYSTEM):,} chars")
print(f"   Phase 3 system: {len(SYNTHESIS_SYSTEM):,} chars")
print(f"   Phase 4 system: {len(EVALUATOR_SYSTEM):,} chars")

✅ Prompt templates defined
   Phase 1 system: 651 chars
   Phase 3 system: 870 chars
   Phase 4 system: 1,048 chars


## 6. Pipeline execution

The `PipelineResult` dataclass collects all outputs. `run_pipeline()` orchestrates all four phases.

In [7]:
@dataclass
class PipelineResult:
    context_json: str = ""
    context_parsed: dict | None = None
    priority_plans: dict[str, str] = field(default_factory=dict)
    synthesized_plan: str = ""
    eval_history: list[dict[str, Any]] = field(default_factory=list)
    final_plan: str = ""
    total_elapsed: float = 0.0
    iterations_used: int = 0


async def run_pipeline() -> PipelineResult:
    result = PipelineResult()
    pipeline_start = time.perf_counter()

    async with httpx.AsyncClient(timeout=httpx.Timeout(HTTP_TIMEOUT)) as client:

        # PHASE 1: Prompt Chaining - Parse scenario
        print("=" * 64)
        print("PHASE 1: Parse Scenario  |  Pattern: Prompt Chaining")
        print("=" * 64)

        result.context_json = await call_llm(
            client, "parse", PHASE1_SYSTEM, f"Scenario:\n{SCENARIO}"
        )

        try:
            result.context_parsed = json.loads(result.context_json)
            print(f"  \U0001f4cb Parsed keys: {list(result.context_parsed.keys())}")
        except json.JSONDecodeError as e:
            print(f"  \u26a0\ufe0f  JSON parse warning: {e}. Proceeding with raw text.")

        # PHASE 2: Parallelization - 4 priorities concurrently
        print(f"\n{'=' * 64}")
        print("PHASE 2: Priority Plans  |  Pattern: Parallelization")
        print("=" * 64)

        async def run_priority(role, priority, description):
            system, user = make_priority_prompt(priority, description, result.context_json)
            plan = await call_llm(client, role, system, user)
            return role, plan

        tasks = [
            run_priority(role, pri, desc)
            for role, (pri, desc) in PRIORITY_CONFIGS.items()
        ]

        phase2_start = time.perf_counter()
        plans = await asyncio.gather(*tasks)
        phase2_elapsed = time.perf_counter() - phase2_start

        for role, plan in plans:
            result.priority_plans[role] = plan

        sequential_est = phase2_elapsed * len(plans)
        print(f"\n  \u26a1 Parallel wall time: {phase2_elapsed:.1f}s"
              f" (sequential estimate: ~{sequential_est:.0f}s)")

        # PHASE 3: Prompt Chaining - Synthesize + alternatives
        print(f"\n{'=' * 64}")
        print("PHASE 3: Synthesize      |  Pattern: Prompt Chaining")
        print("=" * 64)

        synthesis_input = (
            f"City context:\n{result.context_json}\n\n"
            + "\n\n".join(
                f"--- {role.upper()} PLAN ---\n{plan}"
                for role, plan in result.priority_plans.items()
            )
        )

        result.synthesized_plan = await call_llm(
            client, "synthesize", SYNTHESIS_SYSTEM, synthesis_input
        )

        # PHASE 4: Evaluator-Optimizer loop
        print(f"\n{'=' * 64}")
        print(f"PHASE 4: Eval Loop       |  Pattern: Evaluator-Optimizer"
              f" (max {MAX_EVAL_ITERATIONS} iters)")
        print("=" * 64)

        current_plan = result.synthesized_plan

        for iteration in range(1, MAX_EVAL_ITERATIONS + 1):
            print(f"\n  -- Iteration {iteration}/{MAX_EVAL_ITERATIONS} --")

            eval_response = await call_llm(
                client, "evaluator", EVALUATOR_SYSTEM,
                f"Plan to evaluate:\n{current_plan}",
            )

            try:
                eval_data = json.loads(eval_response)
                total_passed = eval_data.get("total_passed", 0)
                feedback = eval_data.get("feedback", "")
                checks = eval_data.get("checks", {})

                for name, check in checks.items():
                    icon = "\u2705" if check.get("pass") else "\u274c"
                    reason = check.get("reason", "")[:80]
                    print(f"    {icon} {name}: {reason}")

                print(f"\n    Score: {total_passed}/{len(checks)}"
                      f" (threshold: {EVAL_PASS_THRESHOLD})")

            except json.JSONDecodeError:
                print("  \u26a0\ufe0f  Evaluator returned non-JSON. Treating as fail.")
                eval_data = {"total_passed": 0, "feedback": eval_response}
                total_passed = 0
                feedback = eval_response

            result.eval_history.append({
                "iteration": iteration,
                "total_passed": total_passed,
                "eval_data": eval_data,
            })

            if total_passed >= EVAL_PASS_THRESHOLD:
                print(f"\n  \U0001f389 Plan PASSED evaluation at iteration {iteration}!")
                result.iterations_used = iteration
                break

            if iteration < MAX_EVAL_ITERATIONS:
                print(f"\n  \U0001f504 Refining plan with evaluator feedback...")
                refinement_prompt = (
                    f"Your previous plan scored {total_passed}/8. "
                    f"Specific feedback:\n\n{feedback}\n\n"
                    f"Previous plan:\n{current_plan}\n\n"
                    "Revise to address ALL feedback. Keep passing sections unchanged. "
                    "Return the complete revised plan."
                )
                current_plan = await call_llm(
                    client, "synthesize", SYNTHESIS_SYSTEM, refinement_prompt
                )
            else:
                print(f"\n  \u26a0\ufe0f  Max iterations reached. Using best available plan.")
                result.iterations_used = iteration

        result.final_plan = current_plan
        result.total_elapsed = time.perf_counter() - pipeline_start

    return result

print("\u2705 Pipeline function defined")

✅ Pipeline function defined


## 7. Run the pipeline

Expect ~2-5 minutes depending on model response times. Phase 2 runs 4 models concurrently.

In [8]:
result = await run_pipeline()

PHASE 1: Parse Scenario  |  Pattern: Prompt Chaining
  ⏳ [parse] Calling Claude Sonnet 4...
  ✅ [parse] Claude Sonnet 4 responded (15.2s, 2,794 chars)
  📋 Parsed keys: ['city_profile', 'climate_risk', 'social_context', 'critical_assumptions', 'key_uncertainties']

PHASE 2: Priority Plans  |  Pattern: Parallelization
  ⏳ [emergency] Calling Gemini 2.5 Pro...
  ⏳ [adaptation] Calling Claude Opus 4...
  ⏳ [retreat] Calling OpenAI o3...
  ⏳ [fiscal] Calling GPT-4o...
  ✅ [fiscal] GPT-4o responded (11.5s, 4,527 chars)
  ✅ [retreat] OpenAI o3 responded (28.2s, 6,180 chars)
  ✅ [emergency] Gemini 2.5 Pro responded (40.8s, 6,317 chars)
  ✅ [adaptation] Claude Opus 4 responded (55.5s, 7,163 chars)

  ⚡ Parallel wall time: 55.6s (sequential estimate: ~222s)

PHASE 3: Synthesize      |  Pattern: Prompt Chaining
  ⏳ [synthesize] Calling Claude Opus 4...
  ✅ [synthesize] Claude Opus 4 responded (80.5s, 9,698 chars)

PHASE 4: Eval Loop       |  Pattern: Evaluator-Optimizer (max 3 iters)

  -- Iterat

## 8. Execution summary

In [9]:
print("=" * 64)
print("PIPELINE EXECUTION SUMMARY")
print("=" * 64)

print(f"\n  Total wall time:   {result.total_elapsed:.1f}s")
print(f"  Eval iterations:   {result.iterations_used}/{MAX_EVAL_ITERATIONS}")
print(f"  Final plan length: {len(result.final_plan):,} chars")

print("\n  Model usage:")
phases = {
    "Phase 1 (parse)":      "parse",
    "Phase 2a (emergency)":  "emergency",
    "Phase 2b (adaptation)": "adaptation",
    "Phase 2c (retreat)":    "retreat",
    "Phase 2d (fiscal)":     "fiscal",
    "Phase 3 (synthesize)":  "synthesize",
    "Phase 4 (evaluator)":   "evaluator",
}
for label, role in phases.items():
    cfg = MODELS[role]
    print(f"    {label:28s} \u2192 {cfg.display_name} ({cfg.provider.value})")

if result.eval_history:
    print("\n  Evaluation progression:")
    for entry in result.eval_history:
        n = entry["total_passed"]
        bar = "\u2588" * n + "\u2591" * (8 - n)
        print(f"    Iteration {entry['iteration']}: [{bar}] {n}/8")

PIPELINE EXECUTION SUMMARY

  Total wall time:   335.9s
  Eval iterations:   3/3
  Final plan length: 13,097 chars

  Model usage:
    Phase 1 (parse)              → Claude Sonnet 4 (anthropic)
    Phase 2a (emergency)         → Gemini 2.5 Pro (google)
    Phase 2b (adaptation)        → Claude Opus 4 (anthropic)
    Phase 2c (retreat)           → OpenAI o3 (openai)
    Phase 2d (fiscal)            → GPT-4o (openai)
    Phase 3 (synthesize)         → Claude Opus 4 (anthropic)
    Phase 4 (evaluator)          → GPT-4o (openai)

  Evaluation progression:
    Iteration 1: [░░░░░░░░] 0/8
    Iteration 2: [░░░░░░░░] 0/8
    Iteration 3: [░░░░░░░░] 0/8


## 9. Inspect phase outputs

### Phase 1: Parsed scenario context

In [10]:
if result.context_parsed:
    print(json.dumps(result.context_parsed, indent=2))
else:
    print(result.context_json[:3000])

{
  "city_profile": {
    "population": 500000,
    "floodplain_pct": 30,
    "budget_operating": 1500000000,
    "budget_capital": 200000000
  },
  "climate_risk": {
    "sea_level_median_2050": 0.6,
    "sea_level_worst_case": 1.2,
    "recent_storm_damage": 1800000000
  },
  "social_context": {
    "housing_crisis": true,
    "aging_infrastructure": true,
    "equity_concerns": "Recent storm damage disproportionately harmed low-income neighborhoods, indicating vulnerability disparities and potential for climate adaptation to exacerbate existing inequities"
  },
  "critical_assumptions": [
    "Discount rate of 3-4% for long-term infrastructure investments",
    "Population growth rate of 0.5-1.5% annually over 30 years",
    "Federal/state funding availability at 40-60% of major infrastructure costs",
    "Insurance penetration rate of 60-80% for residential properties",
    "Construction cost inflation of 3-5% annually",
    "Storm frequency increase of 20-40% over 30-year period",

### Phase 2: Individual priority plans (generated in parallel)

In [11]:
from IPython.display import Markdown, display

for role, plan in result.priority_plans.items():
    cfg = MODELS[role]
    display(Markdown(
        f"---\n### Priority: {role.title()}\n"
        f"*Model: {cfg.display_name} ({cfg.provider.value})*\n\n"
        f"{plan[:5000]}"
        + ("\n\n*[truncated]*" if len(plan) > 5000 else "")
    ))

---
### Priority: Emergency
*Model: Gemini 2.5 Pro (google)*

Of course. As a specialist in emergency management and disaster response, here is the detailed plan for the "Immediate Emergency Preparedness" priority area, tailored to the provided city context.

***

### **Section 2: Immediate Emergency Preparedness**

#### **Preamble: Learning from Crisis**

The recent storm, which inflicted $1.8 billion in damages, was a catastrophic but clarifying event. It exposed critical gaps in our city's ability to warn, evacuate, shelter, and support residents, with the impacts falling most heavily on our low-income neighborhoods and communities of color. This section of the resilience plan is the city's first line of defense. It focuses on building robust, equitable, and people-centric systems that can be implemented immediately to save lives and reduce suffering in the next inevitable event. While long-term physical infrastructure is crucial, a city that cannot protect its people in the short term has failed its most basic duty. This plan prioritizes investments that empower communities and ensure no one is left behind when disaster strikes.

---

### 1. Concrete Interventions & Cost Estimates

Our strategy is built on three mutually reinforcing interventions designed to create a layered defense system, moving from city-wide alerts to neighborhood-level safe havens and hyper-local response teams.

| Intervention                                             | 5-Year Capital Cost | Annual Operating Cost (at full deployment) |
| :------------------------------------------------------- | :------------------ | :----------------------------------------- |
| 1. Hyper-Local, Multi-Channel Early Warning System (EWS) | $15 Million         | $3 Million                                 |
| 2. Community Resilience Hubs & Evacuation Support        | $45 Million         | $2.5 Million                               |
| 3. Decentralized Community Response Capacity             | $5 Million          | $2 Million                                 |
| **Total**                                                | **$65 Million**     | **$7.5 Million**                           |

This represents a 5-year capital investment of **$13M/year**, well within the 25-35% allocation ($50-70M/year) of the city's total capital resilience budget, leaving significant capacity for other priority areas like physical infrastructure.

---

#### **Intervention 1: Hyper-Local, Multi-Channel Early Warning System (EWS)**

*   **Description:** Upgrade the city's generic alert system to a state-of-the-art EWS that provides specific, actionable warnings tailored to individual neighborhoods. The system will integrate real-time inundation models with multi-channel communication platforms.
    *   **Components:** High-resolution flood sensors in vulnerable areas, predictive modeling software, and a communication platform that pushes alerts via SMS, social media, automated voice calls, television/radio overrides, and partnerships with community media outlets. A critical component is a "low-tech" corps of trained "Community Messengers" for door-to-door warnings for residents without reliable phone or internet access.
*   **Equity Focus:** The system will be co-designed with community leaders from low-income neighborhoods. All alerts will be available in the city's top five non-English languages. The Community Messenger program will be staffed by paid, trusted local residents, creating jobs and ensuring the most isolated individuals (e.g., seniors, undocumented residents) are reached.

#### **Intervention 2: Community Resilience Hubs & Evacuation Support Network**

*   **Description:** Identify and retrofit 20-25 existing, trusted public facilities (e.g., schools, libraries, community centers) to serve as "Resilience Hubs." These hubs will be hardened against storm impacts and equipped with backup power (solar + battery storage), emergency communication systems, and pre-positioned supplies (food, water, medical kits, sandbags).
    *   **Function:** During non-emergency times, they serve their normal function. During a storm, they become cooling/warming centers, emergency shelters for those who cannot evacuate, and post-disaster distribution points for aid and recovery services (FEMA registration, insurance claims assistance).
*   **Equity Focus:** At least 75% of these hubs will be located directly within or on the border of census tracts identified as having high social vulnerability. The plan includes funding for a dedicated, accessible evacuation transportation network, using city buses and contracted private vehicles to move residents with mobility challenges from their homes to these hubs or out-of-city shelters.

#### **Intervention 3: Decentralized Community Response Capacity Building**

*   **Description:** Establish and fund a city-wide network of Community Emergency Response Teams (CERTs), with a focus on the 30% of the city located in the floodplain. The city's Office of Emergency Management (OEM) will provide standardized

*[truncated]*

---
### Priority: Adaptation
*Model: Claude Opus 4 (anthropic)*

# Medium-term Infrastructure Adaptation Plan

## Executive Summary
This plan allocates **$60M annually** (30% of total capital budget) for critical infrastructure adaptations over the next 30 years, focusing on protecting essential services while transitioning to resilient systems that can withstand 0.6-1.2m of sea level rise and increased storm frequency.

## Concrete Interventions

### 1. Elevated Critical Infrastructure Corridors ($180M total)
**Description**: Elevate and harden 12 miles of critical roadways connecting hospitals, emergency services, and low-income neighborhoods to evacuation routes.

**Cost Breakdown**:
- Phase 1 (2025-2030): $60M - 4 miles of highest-risk corridors
- Phase 2 (2031-2035): $60M - 4 miles of secondary corridors  
- Phase 3 (2036-2040): $60M - 4 miles of tertiary corridors

**Equity Focus**: Prioritizes corridors serving the 3 lowest-income neighborhoods (median income <$35K) that experienced 65% of recent storm damage.

### 2. Distributed Green-Grey Stormwater Network ($240M total)
**Description**: Hybrid system combining permeable surfaces, bioswales, underground cisterns, and smart pump stations across 150 city blocks.

**Cost Breakdown**:
- Immediate (2025-2027): $40M - Pilot in 25 blocks
- 5-year (2028-2032): $100M - Expand to 75 blocks
- 10-year (2033-2037): $100M - Complete 150 blocks

**Equity Focus**: 60% of installations in Environmental Justice communities, reducing flood insurance premiums by estimated 15-25%.

### 3. Modular Flood Barriers for Critical Facilities ($90M total)
**Description**: Deployable flood protection systems for 45 critical facilities (hospitals, schools, water treatment plants, community centers).

**Cost Breakdown**:
- Immediate (2025-2026): $30M - 15 highest-risk facilities
- 5-year (2027-2031): $30M - 15 secondary facilities
- 10-year (2032-2036): $30M - 15 remaining facilities

**Equity Focus**: Includes all 12 community centers and 8 schools in low-income areas as priority sites.

### 4. Living Shoreline Network ($150M total)
**Description**: 8 miles of oyster reefs, salt marshes, and engineered berms providing 30-50% wave attenuation while creating habitat.

**Cost Breakdown**:
- Phase 1 (2025-2030): $50M - 2.5 miles pilot sections
- Phase 2 (2031-2035): $50M - 3 miles expansion
- Phase 3 (2036-2040): $50M - 2.5 miles completion

**Equity Focus**: Creates 200+ green jobs with training programs targeting residents from affected neighborhoods.

### 5. Smart Infrastructure Monitoring System ($30M total)
**Description**: IoT sensors, predictive analytics, and automated response systems for real-time infrastructure management.

**Cost Breakdown**:
- Immediate (2025-2026): $10M - Core system deployment
- 5-year (2027-2031): $10M - Expansion and integration
- 10-year (2032-2036): $10M - Upgrades and maintenance

**Equity Focus**: Provides free flood alerts via SMS to all residents, with multilingual support.

## Timeline Summary

| Period | Annual Budget | Focus Areas |
|--------|--------------|-------------|
| 2025-2029 | $60M | Critical corridors, pilot projects, immediate protection |
| 2030-2034 | $60M | Network expansion, living shorelines, system integration |
| 2035-2039 | $60M | Completion, upgrades, transition to maintenance |
| 2040-2054 | $40M | Operations, replacements, climate adjustments |

## Funding and Financing Mechanisms

### Capital Stack Structure
1. **Federal Grants (40%)**: $24M/year
   - FEMA BRIC program
   - EPA Water Infrastructure Finance
   - USDOT RAISE grants

2. **Municipal Green Bonds (30%)**: $18M/year
   - 30-year terms at 3.5% interest
   - Climate-certified for ESG investors

3. **State Resilience Funds (20%)**: $12M/year
   - State revolving loan funds
   - Regional climate compacts

4. **Public-Private Partnerships (10%)**: $6M/year
   - Performance-based contracts
   - Insurance industry co-investment

## Governance Structure

### Infrastructure Resilience Board
- **Composition**: 
  - 3 community representatives (1 from each affected district)
  - 2 technical experts (engineering, ecology)
  - 2 city officials (Public Works, Planning)
  - 1 equity advocate

- **Responsibilities**:
  - Quarterly progress reviews
  - Annual budget recommendations
  - Community grievance resolution
  - Performance metric tracking

### Community Engagement Framework
1. **Neighborhood Infrastructure Committees** (monthly meetings)
2. **Youth Climate Corps** (hands-on green infrastructure maintenance)
3. **Multilingual Technical Assistance** (engineering support for residents)
4. **Digital Participation Platform** (real-time project feedback)

## Success Metrics

### Quantitative Targets
| Metric | Baseline (2024) | 5-Year Target | 10-Year Target | 30-Year Target |
|--------|----------------|---------------|----------------|----------------|
| Critical facilities protected | 0 | 15 | 30 | 45 |
| Residents with <10min evacuation access | 250,000 | 350,000 | 425,000 | 475,000 |
| Stormwater capacity (gallons/event) | 50M | 15

*[truncated]*

---
### Priority: Retreat
*Model: OpenAI o3 (openai)*

## Priority Area 3: Long-term Managed Retreat & Land-Use Transformation  

(Capital envelope assumed: **$55 M/yr average (≈28 % of $200 M/yr city cap-ex)**. All dollar figures in 2024 USD.)

---

### 1. Concrete Interventions

| # | Intervention | Scope & Equity Lens | City Share of Capital | External Share | Timing |
|---|--------------|---------------------|-----------------------|----------------|--------|
| 1 | Voluntary Resilience Buy-Out Program (VRBP) | Purchase 4,000 residential parcels (70 % owner-occupied, 30 % rental) in the 100-yr floodplain; guarantee: • Minimum buy-out = pre-storm assessed value • + $25 k “Equity Bonus” for households <$60 k income • 24-mo relocation counselling | $10 M/yr (20 %) | FEMA/HUD 80 % | 2024-2034 |
| 2 | Inland Affordable Resettlement Corridors (ARC) | Up-zone & pre-entitle 600 acres along two transit corridors outside 500-yr floodplain; build 8,000 mixed-income units (60 % ≤ 80 % AMI). 30 % of units reserved for VRBP participants. | $15 M/yr (25 %) | LIHTC + Private 50 %, State bonds 25 % | 2025-2054 |
| 3 | Climate Buffer Land Conversion (CBLC) | Convert 1,000 acres of acquired land to tidal marsh, floodable parks, and solar-canopy community spaces; prioritize parcels adjacent to low-income neighborhoods for co-benefits (heat + air-quality). | $3.5 M/yr (50 %) | NOAA & Philanthropy 50 % | 2026-2054 |
| 4 | Retreated Infrastructure De-commissioning (RID) | Remove/relocate 42 km of roads, 18 km of sewer force mains, 4 substations; jobs program guarantees first hire for impacted residents. | $5 M/yr (30 %) | State resilience bond 70 % | 2028-2054 |
| 5 | Community Land Bank & Stewardship Fund (CLBSF) | Hold retreated parcels in public trust; lease land for community-owned solar/Agrivoltaics; revenues recycle to VRBP. | $0.5 M start-up + self-funding by 2030 | N/A (revenue-positive) | 2024-ongoing |

**30-yr City Capital Outlay ≈ $1.0 B** ➜ **$33.5 M/yr average**, under the $55 M/yr ceiling.

---

### 2. Realistic Timelines

Immediate (2024-2026)  
• Pass Retreat & Transformation Ordinance; set up CLBSF and Inter-agency Retreat Authority (IRA).  
• Pilot VRBP: 300 buy-outs, 120 affordable units delivered.  

5-Year Milestone (2029)  
• 1,500 properties acquired (≥50 % low-income).  
• ARC Phase-1: 2,000 affordable & 1,000 market-rate units online.  
• 150 acres of new buffer parkland open.  

10-Year Milestone (2034)  
• 4,000 total buy-outs complete; population in 100-yr floodplain reduced by 12,000.  
• De-commission 25 % of at-risk utility segments.  
• CLBSF cash-flow–positive; finances 10 % of annual VRBP costs.  

30-Year Horizon (2054)  
• Zero permanent residences within 2050 1-in-100 risk zone.  
• 8,000 affordable & 5,300 market-rate inland units built; no net loss of low-income housing stock.  
• 1,000-acre contiguous coastal greenway absorbs ≥60 % of 1-in-10 storm surge (per hydrodynamic modeling).  

---

### 3. Funding & Financing Toolkit

1. Federal  
   • FEMA Hazard Mitigation Grant Program (up to 90 % for low-income buy-outs)  
   • HUD CDBG-DR & PRO Housing Grants (relocation + ARC)  
2. State  
   • 2025 Coastal Resilience Bond (target: $350 M regionally)  
   • State Infrastructure Bank low-interest loans for RID  
3. Municipal  
   • $40 M green general-obligation bonds every 4 yrs (within $50-100 M bond capacity)  
   • Climate Impact Fee: $2/ft² on new waterfront commercial developments (projected $4 M/yr)  
4. Private / Philanthropic  
   • Community Reinvestment Act credit for local banks financing ARC  
   • $30 M catalytic grant from Coastal Justice Fund for CBLC  
5. Revenue Recycling  
   • Land-lease income from CLBSF solar arrays (est. $1.2 M/yr by 2032)  
   • Insurance premium discounts captured via “Resilience Dividend” district (special assessment)

---

### 4. Governance & Community Engagement

• Establish **Inter-agency Retreat Authority (IRA)** in 2024: reps from Planning, Housing, Public Works, Tribal Nation, two Environmental Justice (EJ) NGOs, and three elected community delegates from affected ZIP codes.  
• **Community Retreat Councils (CRCs)** in each of the five flood-prone neighborhoods. CRCs hold veto-power over parcel sequencing and park designs.  
• **Participatory Budgeting**: 10 % of CLBSF annual surplus allocated by resident vote.  
• Independent **Equity Auditor** (housed at City University) publishes annual scorecards.  
• MOU with regional utilities ensuring that service roll-backs align with IRA timelines to avoid “stranded residents.”

---

### 5. Measurable Success Metrics

1. Properties removed from 100-yr floodplain: 4,000 by 2034; 0 occupied by 2054.  
2. Share of buy-out recipients ≤80 % AMI: ≥60 % each year.  
3. New affordable units built outside hazard zones: 8,000 by 2054.  
4. Acres of restored buffer land: 1,000 acres; vegetative cover ≥70 % survivorship after 5 yrs.  
5. Avoided annual storm-damage cost: ≥$120 M/yr by 2054 (modeled).  
6. Net change in city tax base: ≤5 % decline despite retreat (tr

*[truncated]*

---
### Priority: Fiscal
*Model: GPT-4o (openai)*

# Fiscal Sustainability Plan for Coastal City Resilience

## Overview

The fiscal sustainability priority area aims to ensure that the city maintains robust financial health while addressing climate risks, particularly focusing on equitable outcomes for low-income neighborhoods. This plan outlines strategic interventions, funding mechanisms, governance structures, and success metrics to achieve fiscal sustainability over a 30-year horizon.

## Interventions

### 1. Climate Resilience Bond Program
**Description:** Issue municipal bonds specifically for climate adaptation projects, focusing on flood defenses and infrastructure upgrades in vulnerable areas.

- **Cost Estimate:** $50M annually
- **Timeline:** Immediate (2024) and ongoing
- **Funding Mechanism:** Municipal bonds with a capacity of $50-100M annually
- **Equity Focus:** Prioritize projects in low-income neighborhoods to reduce vulnerability and improve infrastructure.
- **Success Metrics:** Bond issuance of $50M/year, 70% of funds directed to high-risk, low-income areas.

### 2. Property Tax Adjustment and Incentive Program
**Description:** Implement a progressive property tax adjustment in flood-prone areas, combined with incentives for property owners to invest in resilience measures.

- **Cost Estimate:** Administrative costs of $5M over 5 years
- **Timeline:** 5 years (2029)
- **Funding Mechanism:** Adjusted property tax revenues
- **Equity Focus:** Offer tax rebates and grants for resilience improvements to low-income homeowners.
- **Success Metrics:** 20% reduction in tax burden for low-income residents, 30% increase in property resilience investments.

### 3. Public-Private Partnership for Green Infrastructure
**Description:** Develop partnerships with private entities to co-fund green infrastructure projects, leveraging technology cost reductions.

- **Cost Estimate:** $30M annually (public share)
- **Timeline:** 10 years (2033)
- **Funding Mechanism:** Public-private partnerships, federal/state matching funds
- **Equity Focus:** Ensure projects are located in underserved communities, providing job opportunities and environmental benefits.
- **Success Metrics:** $60M total investment annually, 50% of projects in low-income areas.

### 4. Managed Retreat and Buyout Program
**Description:** Facilitate a voluntary buyout program for properties in high-risk flood zones, with a focus on equitable relocation support.

- **Cost Estimate:** $100M over 10 years
- **Timeline:** 10 years (2033)
- **Funding Mechanism:** Federal managed retreat programs (75-90% funding)
- **Equity Focus:** Prioritize buyouts in low-income neighborhoods, provide relocation assistance and affordable housing options.
- **Success Metrics:** 1,000 properties bought out, 70% participation rate in target areas.

### 5. Emergency Preparedness Fund
**Description:** Establish a dedicated fund for emergency preparedness and response, emphasizing community-led initiatives.

- **Cost Estimate:** $20M annually
- **Timeline:** Immediate (2024) and ongoing
- **Funding Mechanism:** Reallocation of existing budget, state grants
- **Equity Focus:** Fund community-based organizations in low-income neighborhoods for localized response efforts.
- **Success Metrics:** 50% of funds allocated to community-led projects, 90% of vulnerable areas with emergency plans.

## Governance and Community Engagement

- **Governance Structure:** Establish a Resilience Finance Committee comprising city officials, community leaders, and financial experts to oversee implementation.
- **Community Engagement:** Conduct regular town hall meetings and workshops to involve residents in decision-making, particularly in low-income areas.
- **Transparency Measures:** Publish annual reports detailing progress, expenditures, and outcomes.

## Social/Ethical Tradeoffs

- **Resource Allocation:** Balancing immediate needs with long-term investments may delay benefits for some communities.
- **Property Tax Adjustments:** Potential resistance from property owners in high-value areas due to increased tax rates.
- **Managed Retreat:** Ethical concerns regarding displacement and community identity loss must be addressed through comprehensive support programs.

## Conclusion

This fiscal sustainability plan aims to create a resilient financial framework that supports equitable climate adaptation efforts. By focusing on targeted interventions and inclusive governance, the city can enhance its resilience while addressing the needs of its most vulnerable populations.

### Phase 3+4: Final synthesized plan (after evaluation loop)

In [12]:
from IPython.display import Markdown, display
display(Markdown(result.final_plan))

# Unified 30-Year Coastal City Resilience Strategy

## Executive Summary

This integrated resilience strategy synthesizes four specialist plans into a cohesive 30-year roadmap for a coastal city of 500,000 facing severe climate risks. With $200M annual capital budget constraints and recent $1.8B storm damage disproportionately affecting low-income communities, this strategy prioritizes equitable adaptation through phased implementation of emergency preparedness, infrastructure hardening, managed retreat, and fiscal sustainability measures.

## 1. Budget Reconciliation and Prioritization

### Initial Budget Requests vs. Available Funding

The four specialist plans initially requested:
- Emergency Preparedness: $13M/year capital + $7.5M operations
- Infrastructure Adaptation: $60M/year capital
- Managed Retreat: $33.5M/year capital
- Fiscal Sustainability: $50M/year bonds + $30M/year PPP

**Total Initial Request: $186.5M/year capital** (within budget)

### Final Allocated Budget (30-Year Average)

| Priority Area | Years 1-10 | Years 11-20 | Years 21-30 | 30-Year Average |
|--------------|------------|-------------|-------------|-----------------|
| Emergency Preparedness | $13M | $5M | $3M | $7M |
| Infrastructure Adaptation | $60M | $60M | $40M | $53.3M |
| Managed Retreat | $25M | $40M | $35M | $33.3M |
| Fiscal Sustainability | $50M | $50M | $50M | $50M |
| Strategic Reserve | $52M | $45M | $72M | $56.4M |
| **Total Annual Capital** | **$200M** | **$200M** | **$200M** | **$200M** |

### Budget Rationale

1. **Front-loaded emergency preparedness** to save lives immediately
2. **Sustained infrastructure investment** through year 20, then reduced as assets reach design life
3. **Ramped retreat program** as sea levels rise and federal funding becomes available
4. **Consistent fiscal measures** to maintain revenue base
5. **Strategic reserve** for catastrophic events and emerging opportunities

## 2. Cross-Cutting Synergies

### Integrated Investment Opportunities

1. **Resilience Hubs as Multi-Use Infrastructure**
   - Emergency shelters (Emergency Plan)
   - Green infrastructure nodes (Adaptation Plan)
   - Community land trust anchors (Retreat Plan)
   - Revenue generators through solar installations (Fiscal Plan)

2. **Green-Grey Infrastructure Network**
   - Stormwater management (Adaptation)
   - Emergency evacuation routes (Emergency)
   - Property value stabilization (Fiscal)
   - Buffer zones for retreat areas (Retreat)

3. **Community Response Teams**
   - Emergency first responders (Emergency)
   - Green infrastructure maintenance crews (Adaptation)
   - Retreat counseling support (Retreat)
   - Local job creation (Fiscal)

4. **Data and Monitoring Systems**
   - Early warning systems (Emergency)
   - Infrastructure performance tracking (Adaptation)
   - Property value monitoring (Retreat/Fiscal)
   - Revenue impact assessment (Fiscal)

### Synergy Value Capture

These integrated approaches generate approximately 30% cost savings through:
- Shared capital investments ($20M/year saved)
- Reduced operating costs through multi-use facilities ($5M/year)
- Enhanced federal grant competitiveness (additional $15M/year)

## 3. Unified Implementation Timeline

### Phase 1: Immediate Protection (2024-2029)
**Focus:** Life safety and critical system protection

- **Year 1-2:** Launch early warning system pilots, identify resilience hub sites, establish governance structures
- **Year 3-4:** Complete first 5 resilience hubs, begin critical corridor elevation, pilot voluntary buyouts
- **Year 5:** Full early warning system deployment, 15 hubs operational, 300 properties acquired

**Budget:** Heavy emergency and infrastructure focus (40% and 30% respectively)

**Equity Targets:** 
- 80% of resilience hubs located in neighborhoods with median household income <$40,000 (Riverside, Eastside, and South Harbor districts)
- 100% of early warning systems deployed first in communities where >60% of residents lack flood insurance
- Priority buyouts for households earning <50% Area Median Income ($35,000) with 150% fair market value offers

### Phase 2: Systematic Hardening (2030-2039)
**Focus:** Infrastructure transformation and accelerated retreat

- **Year 6-10:** Complete critical infrastructure elevation, expand green-grey network to 75 blocks
- **Year 11-15:** Living shoreline construction, major utility relocations, 2,000 buyouts complete

**Budget:** Balanced across all priorities, retreat acceleration (20% to 25%)

**Equity Targets:**
- Green infrastructure investments: 65% in Environmental Justice communities (census tracts with >40% poverty rate)
- Workforce development: 50% of green jobs to residents from zip codes 12345, 12346, 12348 (highest unemployment)
- Retreat assistance: Households <80% AMI receive additional $50,000 relocation assistance

### Phase 3: Transformation (2040-2054)
**Focus:** Complete transition to resilient configuration

- **Year 16-20:** Complete infrastructure systems, 4,000 buyouts achieved
- **Year 21-30:** Maintain and adapt systems, complete land use transformation

**Budget:** Reduced infrastructure needs, sustained retreat and maintenance

**Equity Targets:**
- No low-income household (<$50,000) pays >5% of income for flood insurance
- 90% of former floodplain residents relocated within same school district
- Community land trusts preserve affordability for 2,000 units in receiving neighborhoods

## 4. Alternative Strategies Considered

### Alternative A: "Fortress City"
**Description:** Massive sea walls and pump systems protecting entire 30% floodplain

**Pros:**
- Protects all existing development
- Maintains current tax base
- Simpler governance

**Cons:**
- $500M+ capital cost
- High maintenance burden
- Catastrophic failure risk
- No co-benefits

**Rejection Rationale:** Financially infeasible, inequitable (protects wealthy waterfront over inland poor), and creates false security leading to increased development in risk zones.

### Alternative B: "Rapid Full Retreat"
**Description:** Mandatory buyouts of all floodplain properties within 10 years

**Pros:**
- Eliminates flood risk
- Clear end state
- Maximum federal funding eligibility

**Cons:**
- $2B+ total cost
- Massive social disruption
- Political impossibility
- Destroys communities

**Rejection Rationale:** Socially unjust, financially impossible even with federal support, and would devastate low-income communities while wealthy areas could rebuild elsewhere.

### Alternative C: "Market-Led Adaptation"
**Description:** Minimal public investment, rely on insurance markets and private adaptation

**Pros:**
- Low public cost
- Market efficiency
- Individual choice

**Cons:**
- Abandons low-income residents
- Fragmented response
- Inadequate for systemic risk
- Increases inequality

**Rejection Rationale:** Recent storm showed market failures disproportionately harm vulnerable populations; public intervention essential for equity.

## 5. Leading Indicators for Strategy Switching

Monitor these indicators quarterly; if 3+ trigger, convene emergency strategy review:

1. **Sea Level Rise Acceleration**
   - Trigger: Observed SLR exceeds 2cm/year for 3 consecutive years
   - Response: Accelerate retreat timeline, reduce infrastructure investments in Zone A

2. **Federal Funding Collapse**
   - Trigger: Federal climate funding falls below $10M/year for 2 years
   - Response: Shift to Alternative C with enhanced local revenue measures

3. **Catastrophic Storm Damage**
   - Trigger: Single event causes >$3B damage or >50 fatalities
   - Response: Emergency pivot to Fortress City for critical areas only

4. **Community Resistance**
   - Trigger: <40% participation in voluntary programs for 2 consecutive years
   - Response: Enhance incentives, slow timeline, increase engagement budget 100%

5. **Fiscal Crisis**
   - Trigger: City bond rating drops below BBB or tax base erodes >30%
   - Response: Suspend new capital projects, focus on life safety only

## 6. Critical Local Datasets for Uncertainty Reduction

### Priority Data Collection Initiatives

1. **High-Resolution Elevation and Flood Modeling**
   - Current gap: 10m resolution, 2015 vintage
   - Need: 1m resolution, annual updates
   - Cost: $2M initial, $200k/year
   - Impact: Could shift 20% of properties between risk categories, saving $50M in unnecessary interventions

2. **Real-Time Property Transaction and Value Database**
   - Current gap: Annual assessments, 18-month lag
   - Need: Monthly updates linked to flood risk
   - Cost: $500k system, $100k/year
   - Impact: Enable dynamic retreat prioritization, optimize buyout timing for 30% cost reduction

3. **Community Social Network Mapping**
   - Current gap: No systematic data on neighborhood cohesion
   - Need: Block-level social capital indices
   - Cost: $300k study, $50k/year updates
   - Impact: Improve retreat participation by 40% through cohort-based relocation

4. **Hyperlocal Weather Station Network**
   - Current gap: 3 stations citywide
   - Need: 50 stations in vulnerable areas
   - Cost: $1M installation, $200k/year
   - Impact: Reduce false alarm rates by 60%, increase warning lead time by 2 hours

5. **Green Infrastructure Performance Monitoring**
   - Current gap: No systematic effectiveness data
   - Need: IoT sensors on all installations
   - Cost: $50k per installation
   - Impact: Optimize designs for 50% better performance at same cost

6. **Insurance Coverage and Claims Database**
   - Current gap: No unified view of coverage gaps
   - Need: Integrated public-private database
   - Cost: $200k development, $50k/year
   - Impact: Target assistance to reduce uninsured losses by 70%

### Data Governance Framework

- Establish Chief Resilience Data Officer position
- Create public data portal with privacy protections
- Mandate data sharing agreements with utilities and insurers
- Quarterly data quality audits and community accessibility reviews

## 7. Key Assumptions and Uncertainties

### Explicit Planning Assumptions

1. **Climate Projections**
   - Sea level rise: 2-4 feet by 2054 (medium confidence, ±1 foot uncertainty)
   - Storm intensity: 20-30% increase in Category 3+ storms (low confidence, ±15%)
   - Precipitation: 15-25% increase in extreme rainfall events (medium confidence, ±10%)

2. **Economic Assumptions**
   - Federal funding: $20-40M/year available (high uncertainty, could range $0-100M)
   - Construction costs: 3-5% annual inflation (medium confidence, ±2%)
   - Property values: 2-3% annual growth in safe areas, -5% to -10% in flood zones (high uncertainty)

3. **Social Assumptions**
   - Voluntary retreat participation: 60-80% with adequate incentives (medium confidence, ±20%)
   - Population growth: 0.5-1% annually (high confidence, ±0.3%)
   - Insurance availability: Private market retreat from highest-risk areas by 2035 (high confidence)

4. **Technical Assumptions**
   - Green infrastructure effectiveness: 40-60% runoff reduction (medium confidence, ±15%)
   - Early warning accuracy: 85-95% for 24-hour forecasts (high confidence, ±5%)
   - Infrastructure lifespan: 30-50 years for elevated structures (medium confidence, ±10 years)

### Uncertainty Quantification

**High Uncertainty Factors** (could change strategy fundamentally):
- Federal climate policy and funding (40% probability of major shifts)
- Breakthrough adaptation technologies (20% probability of game-changing innovation)
- Cascading infrastructure failures (15% probability of system-wide collapse)

**Medium Uncertainty Factors** (would modify implementation):
- Community acceptance rates (30% probability of significant resistance)
- Regional economic conditions (35% probability of recession impacting funding)
- Insurance market stability (40% probability of major disruptions)

**Low Uncertainty Factors** (unlikely to change core strategy):
- Basic climate science (5% probability of major projection revisions)
- Engineering standards (10% probability of significant changes)
- Demographic trends (15% probability of unexpected shifts)

### Adaptive Management Framework

- Annual assumption review with expert panel
- Biennial strategy adjustments based on observed vs. projected outcomes
- Major strategy review every 5 years or when 2+ high uncertainty factors shift
- Continuous scenario planning for high-impact, low-probability events

## Conclusion

This unified strategy creates a feasible path to urban resilience within budget constraints while centering equity. By integrating emergency preparedness, infrastructure adaptation, managed retreat, and fiscal sustainability, the city can protect lives, reduce damages, and transform itself for a climate-altered future. The phased approach allows for learning and adjustment while maintaining momentum toward a safer, more equitable coastal community.

The strategy's success depends on sustained political will, community engagement, and adaptive management as new data emerges. With careful implementation and regular reassessment, this framework positions the city to thrive despite rising seas and intensifying storms.

### Phase 4: Evaluation history

In [13]:
for entry in result.eval_history:
    print(f"\n-- Iteration {entry['iteration']} -- Score: {entry['total_passed']}/8")
    checks = entry["eval_data"].get("checks", {})
    for name, check in checks.items():
        icon = "\u2705" if check.get("pass") else "\u274c"
        print(f"  {icon} {name}: {check.get('reason', '')}")

    feedback = entry["eval_data"].get("feedback", "")
    if feedback and entry["total_passed"] < EVAL_PASS_THRESHOLD:
        print(f"\n  Feedback to generator:\n  {feedback[:500]}")


-- Iteration 1 -- Score: 0/8

  Feedback to generator:
  ```json
{
  "checks": {
    "BUDGET_FEASIBILITY": {
      "pass": true,
      "reason": "Total capital costs are $200M/year, which is within the budget constraint."
    },
    "EQUITY_SPECIFICITY": {
      "pass": false,
      "reason": "Equity impacts are mentioned, but specific neighborhoods, income thresholds, or percentages are not detailed."
    },
    "COST_CONCRETENESS": {
      "pass": true,
      "reason": "Cost estimates are provided as order-of-magnitude numbers (e.g., $XM)."
    },
 

-- Iteration 2 -- Score: 0/8

  Feedback to generator:
  ```json
{
    "checks": {
        "BUDGET_FEASIBILITY": {
            "pass": true,
            "reason": "Total capital costs are within the $200M/year budget constraint."
        },
        "EQUITY_SPECIFICITY": {
            "pass": true,
            "reason": "Equity impacts are specific with named neighborhoods, income thresholds, and percentages."
        },
        "COST_CO

## 10. Export results

In [14]:
output_dir = "pipeline_output"
os.makedirs(output_dir, exist_ok=True)

with open(f"{output_dir}/final_resilience_plan.md", "w") as f:
    f.write("# 30-Year Coastal City Resilience & Equity Plan\n\n")
    f.write(f"*Generated by multi-provider agentic pipeline*\n")
    f.write(f"*Evaluation iterations: {result.iterations_used}*\n\n---\n\n")
    f.write(result.final_plan)

with open(f"{output_dir}/parsed_context.json", "w") as f:
    try:
        json.dump(json.loads(result.context_json), f, indent=2)
    except json.JSONDecodeError:
        f.write(result.context_json)

for role, plan in result.priority_plans.items():
    with open(f"{output_dir}/priority_{role}.md", "w") as f:
        cfg = MODELS[role]
        f.write(f"# {role.title()} -- {cfg.display_name}\n\n{plan}")

with open(f"{output_dir}/eval_history.json", "w") as f:
    json.dump(result.eval_history, f, indent=2, default=str)

print(f"\U0001f4c1 Saved to {output_dir}/:")
for fname in sorted(os.listdir(output_dir)):
    size = os.path.getsize(f"{output_dir}/{fname}")
    print(f"   {fname:40s} {size:>8,} bytes")

📁 Saved to pipeline_output/:
   eval_history.json                           5,470 bytes
   final_resilience_plan.md                   13,235 bytes
   parsed_context.json                         2,794 bytes
   priority_adaptation.md                      7,194 bytes
   priority_emergency.md                       6,348 bytes
   priority_fiscal.md                          4,547 bytes
   priority_retreat.md                         6,319 bytes


## 11. Pattern analysis & observations

### Why this combination of patterns works

| Pattern | Where used | Benefit |
|---------|-----------|---------|
| **Prompt chaining** | Phase 1 to 2, Phase 3 to 4 | Each step gets focused context; gates catch budget overruns early |
| **Parallelization** | Phase 2 (4 priorities) | ~4x latency reduction; each model specializes in its domain |
| **Evaluator-optimizer** | Phase 4 | Catches the specific failure modes LLMs exhibit on this prompt |
| **Routing** | Phase 2 (implicit) | Hard reasoning (retreat/ethics) to o3; structured work to GPT-4o |

### Key design decisions

1. **Cross-provider evaluator**: GPT-4o evaluates Claude Opus 4's synthesis to avoid self-reinforcement bias
2. **Structured handoff**: Phase 1 produces JSON that all Phase 2 workers consume, preventing models from interpreting scenario numbers differently
3. **Budget gate**: The evaluator's first rubric check (BUDGET_FEASIBILITY) catches the most common LLM failure on this prompt
4. **Max 3 iterations**: Diminishing returns beyond 2-3 refinement cycles; most plans pass by iteration 2

### Extending this pipeline

- Add **web search** to Phase 2 workers for real cost benchmarks (FEMA data, USACE project costs)
- Add a **routing** classifier before Phase 2 to dynamically assign models based on query complexity
- Run the **voting** variant of parallelization: have 2 models produce each priority plan, pick the better one